In [10]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Project root: StockLens/
PROJECT_ROOT = Path.cwd().parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection.mrmr import mrmr_greedy

feature_target_mi_path = "../../data/processed/filter_results/feature_target_mi.csv"
feature_mi_matrix_path = "../../data/processed/filter_results/feature_mi_matrix.csv"

feature_target_mi = pd.read_csv(feature_target_mi_path)
feature_mi_matrix = pd.read_csv(feature_mi_matrix_path)

print("Feature-Target MI")
display(feature_target_mi.head())

print("Feature MI Matrix")
display(feature_mi_matrix.head())

Feature-Target MI


,feature,mi_score
0,sma_5,0.092133
1,sma_20,0.075937
2,atr_14,0.073692
3,sma_60,0.070101
4,volatility_20,0.068883


Feature MI Matrix


,Unnamed: 0,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,sma_20,...,roc_20,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20
0,return_1d,0.000000,0.139045,0.072892,0.064534,0.590836,0.201050,0.272274,0.267512,0.168916,...,0.064534,0.047641,0.047901,0.024838,0.180285,0.116514,0.086687,0.083187,0.087893,0.115599
1,return_5d,0.139045,0.000000,0.368003,0.201841,0.063755,0.078634,0.103450,0.100724,0.084446,...,0.201841,0.129589,0.077677,0.302843,0.174976,0.121862,0.102379,0.000000,0.094834,0.113060
2,return_10d,0.072892,0.368003,0.000000,0.436106,0.079212,0.129943,0.067880,0.153295,0.144588,...,0.436106,0.291461,0.172214,0.613728,0.115350,0.222376,0.195440,0.000000,0.155978,0.060745
3,return_20d,0.064534,0.201841,0.436106,0.000000,0.038361,0.075104,0.052645,0.277313,0.246194,...,6.630282,0.715967,0.481400,0.286162,0.161803,0.347585,0.289020,0.000000,0.278970,0.051378
4,intraday_return,0.590836,0.063755,0.079212,0.038361,0.000000,0.530469,0.052903,0.274167,0.164524,...,0.037755,0.029036,0.022327,0.040020,0.110814,0.061211,0.095628,0.034579,0.061496,0.108304


In [11]:
selected_features = mrmr_greedy(
    feature_target_mi=feature_target_mi,
    feature_mi_matrix=feature_mi_matrix,
    k=10
)

print("Selected Features:")
for i, feature in enumerate(selected_features, start=1):
    print(f"{i}. {feature}")

Selected Features:
1. sma_5
2. volume_change_1d
3. price_to_sma_5
4. roc_20
5. volume_ratio_20
6. volatility_5
7. gap
8. high_low_range
9. rsi_14
10. intraday_return


In [12]:
mrmr_results = pd.DataFrame({
    "rank": range(1, len(selected_features) + 1),
    "feature": selected_features
})

mrmr_results.to_csv(
    "../../data/processed/filter_results/mrmr_results.csv",
    index=False
)

In [13]:
selected_features = mrmr_greedy(
    feature_target_mi=feature_target_mi,
    feature_mi_matrix=feature_mi_matrix,
    k=10
)

relevance = dict(
    zip(
        feature_target_mi["feature"],
        feature_target_mi["mi_score"]
    )
)

mi_matrix = feature_mi_matrix.set_index(
    feature_mi_matrix.columns[0]
)

validation_results = []

for rank, feature in enumerate(selected_features, start=1):

    if rank == 1:
        redundancy = 0.0
    else:
        previous_features = selected_features[:rank - 1]

        redundancy = np.mean([
            mi_matrix.loc[feature, prev]
            for prev in previous_features
        ])

    mrmr_score = relevance[feature] - redundancy

    validation_results.append({
        "rank": rank,
        "feature": feature,
        "relevance": relevance[feature],
        "redundancy": redundancy,
        "mrmr_score": mrmr_score
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

,rank,feature,relevance,redundancy,mrmr_score
0,1,sma_5,0.092133,0.000000,0.092133
1,2,volume_change_1d,0.022044,0.000000,0.022044
2,3,price_to_sma_5,0.013704,0.031251,-0.017546
3,4,roc_20,0.039974,0.146140,-0.106167
4,5,volume_ratio_20,0.006572,0.101383,-0.094810
5,6,volatility_5,0.040760,0.131235,-0.090475
6,7,gap,0.033707,0.157762,-0.124055
7,8,high_low_range,0.014721,0.130922,-0.116201
8,9,rsi_14,0.053273,0.182798,-0.129525
9,10,intraday_return,0.000000,0.159245,-0.159245
